In [1]:
# ============================================
# 第一步：安装缺失的库（RunPod 需要）
# ============================================
!pip install pandas tqdm numpy rouge_score bert_score -q

print("✅ 依赖安装完成")


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
✅ 依赖安装完成


In [2]:
# ============================================
# 第二步：导入所有库
# ============================================
import os
import sys
import zipfile
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from dataclasses import asdict

print("✅ 所有库导入成功")

✅ 所有库导入成功


In [3]:
# 降级 transformers 到兼容版本
!pip uninstall transformers tokenizers -y
!pip install transformers==4.38.2 tokenizers==0.15.2 -q

print("✅ 降级完成！请点击 Runtime -> Restart kernel 重启内核")
print("重启后重新运行下面的完整代码")

Found existing installation: transformers 5.7.0
Uninstalling transformers-5.7.0:
  Successfully uninstalled transformers-5.7.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
✅ 降级完成！请点击 Runtime -> Restart kernel 重启内核
重启后重新运行下面的完整代码


In [1]:
# ============================================
# RunPod 全量推理代码（3396 条数据）
# 基于你的代码修改
# ============================================

import os
import sys
import zipfile
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from dataclasses import asdict
import gc

# 验证 transformers 版本
import transformers
print(f"Transformers version: {transformers.__version__}")
assert transformers.__version__ <= "4.38.2", "请先运行降级 cell 并重启内核"

# ============================================
# 设置路径
# ============================================
DATA_PATH = 'multiclinsum_test_en.zip'
OUTPUT_DIR = './results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

# 检查数据文件
if not os.path.exists(DATA_PATH):
    print(f"\n⚠️ 文件不存在: {DATA_PATH}")
    print(f"当前目录内容: {os.listdir('.')}")
    raise FileNotFoundError(f"找不到 {DATA_PATH}")

# ============================================
# 克隆 dLLM-Cache
# ============================================
print("\n" + "="*60)
print("Cloning dLLM-Cache repository")
print("="*60)

if not os.path.exists('dLLM-cache'):
    !git clone https://github.com/maomaocun/dLLM-cache.git

sys.path.append('dLLM-cache')

try:
    from dllm_cache.cache import dLLMCache, dLLMCacheConfig
    from dllm_cache.hooks import register_cache_LLaDA
    print("✅ dLLM-Cache imported")
except Exception as e:
    print(f"⚠️ dLLM-Cache 导入失败: {e}")
    class dLLMCache:
        @staticmethod
        def new_instance(*args, **kwargs): pass
        def reset_cache(self, *args, **kwargs): pass
    dLLMCacheConfig = lambda **kwargs: type('Config', (), kwargs)()
    def register_cache_LLaDA(*args, **kwargs): pass

# ============================================
# generate 函数
# ============================================
def add_gumbel_noise(logits, temperature):
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel_noise = (- torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise

def get_num_transfer_tokens(mask_index, steps):
    mask_num = mask_index.sum(dim=1, keepdim=True)
    base = mask_num // steps
    remainder = mask_num % steps
    num_transfer_tokens = torch.zeros(mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64) + base
    for i in range(mask_num.size(0)):
        num_transfer_tokens[i, :remainder[i]] += 1
    return num_transfer_tokens

@torch.no_grad()
def generate(model, prompt, attention_mask=None, steps=128, gen_length=128, block_length=128, temperature=0.,
             cfg_scale=0., remasking='low_confidence', mask_id=126336,
             logits_eos_inf=False, confidence_eos_eot_inf=False):
    x = torch.full((prompt.shape[0], prompt.shape[1] + gen_length), mask_id, dtype=torch.long).to(model.device)
    x[:, :prompt.shape[1]] = prompt.clone()

    if attention_mask is not None:
        attention_mask = torch.cat([attention_mask, torch.ones((prompt.shape[0], gen_length), dtype=attention_mask.dtype, device=model.device)], dim=-1)

    prompt_index = (x != mask_id)
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert steps % num_blocks == 0
    steps = steps // num_blocks

    for num_block in range(num_blocks):
        block_mask_index = (x[:, prompt.shape[1] + num_block * block_length: prompt.shape[1] + (num_block + 1) * block_length:] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps)
        for i in range(steps):
            mask_index = (x == mask_id)
            if cfg_scale > 0.:
                un_x = x.clone()
                un_x[prompt_index] = mask_id
                x_ = torch.cat([x, un_x], dim=0)
                if attention_mask is not None:
                    attention_mask_ = torch.cat([attention_mask, attention_mask], dim=0)
                logits = model(x_, attention_mask=attention_mask_).logits
                logits, un_logits = torch.chunk(logits, 2, dim=0)
                logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
            else:
                logits = model(x, attention_mask=attention_mask).logits

            if logits_eos_inf:
                logits[:, :, 126081] = -torch.inf

            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1)

            if confidence_eos_eot_inf:
                logits_with_noise[:, :, 126081] = logits[:, :, 126348] = -torch.inf

            if remasking == 'low_confidence':
                p = F.softmax(logits, dim=-1)
                x0_p = torch.squeeze(torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1)
            elif remasking == 'random':
                x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            else:
                raise NotImplementedError(remasking)

            x0_p[:, prompt.shape[1] + (num_block + 1) * block_length:] = -np.inf
            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_p, -np.inf)

            transfer_index = torch.zeros_like(x0, dtype=torch.bool, device=x0.device)
            for j in range(confidence.shape[0]):
                _, select_index = torch.topk(confidence[j], k=num_transfer_tokens[j, i])
                transfer_index[j, select_index] = True
            x[transfer_index] = x0[transfer_index]
    return x

# ============================================
# 摘要生成函数
# ============================================
def summarize_clinical_text(model, tokenizer, full_text, device, max_length=128):
    prompt = f"""Please generate a concise clinical summary based on the following medical dialogue:

{full_text}

Clinical Summary:"""

    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    encoded = tokenizer(
        [formatted_prompt],
        add_special_tokens=False,
        padding=True,
        return_tensors="pt"
    )
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    try:
        dLLMCache().reset_cache(input_ids.shape[1])
    except:
        pass

    out = generate(
        model,
        input_ids,
        attention_mask,
        steps=64,
        gen_length=max_length,
        block_length=16,
        temperature=0.,
        cfg_scale=0.,
        remasking='low_confidence'
    )

    summary = tokenizer.batch_decode(
        out[:, input_ids.shape[1]:],
        skip_special_tokens=True
    )[0]

    return summary.strip()

# ============================================
# Step 1: 加载数据
# ============================================
print("\n" + "="*60)
print("Step 1: Loading test data")
print("="*60)

records = []
with zipfile.ZipFile(DATA_PATH, 'r') as z:
    files = [f for f in z.namelist() if '/fulltext/' in f and f.endswith('.txt')]
    print(f"Found {len(files)} files")

    for file in tqdm(files, desc="Loading"):
        full_text = z.read(file).decode('utf-8')
        summary_file = file.replace('/fulltext/', '/summaries/').replace('.txt', '_sum.txt')
        summary_text = z.read(summary_file).decode('utf-8')
        records.append({'Full_Text': full_text, 'Summary': summary_text})

df_test = pd.DataFrame(records)
print(f"✅ Loaded {len(df_test)} test samples")

# ============================================
# Step 2: 加载模型
# ============================================
print("\n" + "="*60)
print("Step 2: Loading LLaDA model")
print("="*60)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.cuda.empty_cache()

print("Loading LLaDA-8B-Instruct...")
model = AutoModel.from_pretrained(
    'GSAI-ML/LLaDA-8B-Instruct',
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    'GSAI-ML/LLaDA-8B-Instruct',
    trust_remote_code=True
)

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'

print("✅ Model loaded successfully")

# 初始化 dLLM-Cache
try:
    dLLMCache.new_instance(
        **asdict(
            dLLMCacheConfig(
                prompt_interval_steps=100,
                gen_interval_steps=7,
                transfer_ratio=0.25,
            )
        )
    )
    register_cache_LLaDA(model, "model.transformer.blocks")
    print("✅ dLLM-Cache acceleration enabled")
except:
    print("⚠️ dLLM-Cache not available")

# ============================================
# Step 3: 全量推理
# ============================================
print("\n" + "="*60)
print("Step 3: Full inference on all samples")
print("="*60)

total_samples = len(df_test)
print(f"Total samples to process: {total_samples}")

results = []
start_idx = 0

# 检查是否有之前保存的检查点
checkpoint_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('checkpoint_') and f.endswith('.csv')]
if checkpoint_files:
    latest_checkpoint = sorted(checkpoint_files)[-1]
    checkpoint_df = pd.read_csv(os.path.join(OUTPUT_DIR, latest_checkpoint))
    results = checkpoint_df.to_dict('records')
    start_idx = len(results)
    print(f"✅ Resuming from checkpoint: {start_idx}/{total_samples} already done")

# 分批处理
batch_size = 1  # 单条处理，避免显存峰值

for idx in tqdm(range(start_idx, total_samples), desc="Generating summaries"):
    row = df_test.iloc[idx]
    full_text = row['Full_Text']
    reference_summary = row['Summary']
    
    try:
        predicted_summary = summarize_clinical_text(
            model, tokenizer, full_text, device, max_length=128
        )
    except Exception as e:
        print(f"\n❌ Error on sample {idx}: {e}")
        predicted_summary = ""
    
    results.append({
        'id': idx,
        'predicted_summary': predicted_summary,
        'reference_summary': reference_summary
    })
    
    # 每 100 条保存检查点
    if (idx + 1) % 100 == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv(f'{OUTPUT_DIR}/checkpoint_{idx+1}.csv', index=False)
        print(f"\n📁 Checkpoint saved: {idx+1}/{total_samples}")
        
        # 清理显存
        torch.cuda.empty_cache()
        gc.collect()

# ============================================
# Step 4: 保存最终结果
# ============================================
print("\n" + "="*60)
print("Step 4: Saving final results")
print("="*60)

results_df = pd.DataFrame(results)
csv_path = f'{OUTPUT_DIR}/llada_summaries_full.csv'
results_df.to_csv(csv_path, index=False)
print(f"✅ Full results saved to {csv_path}")

# 统计信息
print("\n" + "="*60)
print("Summary Statistics")
print("="*60)
print(f"Total samples processed: {len(results_df)}")
successful = sum(1 for r in results if r['predicted_summary'])
print(f"Successful generations: {successful}/{len(results_df)}")
avg_len = results_df['predicted_summary'].str.len().mean()
print(f"Average summary length: {avg_len:.0f} chars")

if len(results) > 0 and results[0]['predicted_summary']:
    print(f"\n【Sample Output】:")
    print(f"Reference: {results[0]['reference_summary'][:200]}...")
    print(f"Generated: {results[0]['predicted_summary'][:300]}...")

print("\n✅ All done!")

Transformers version: 4.38.2
✅ Output directory: ./results

Cloning dLLM-Cache repository
✅ dLLM-Cache imported

Step 1: Loading test data
Found 3396 files


Loading: 100%|██████████| 3396/3396 [00:00<00:00, 13811.55it/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Loaded 3396 test samples

Step 2: Loading LLaDA model
Using device: cuda
GPU: NVIDIA L40S
GPU Memory: 47.8 GB
Loading LLaDA-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...


📁 Checkpoint saved: 3200/3396


Generating summaries:  97%|█████████▋| 3300/3396 [2:44:34<05:19,  3.32s/it]


📁 Checkpoint saved: 3300/3396


Generating summaries: 100%|██████████| 3396/3396 [2:49:26<00:00,  2.99s/it]


Step 4: Saving final results
✅ Full results saved to ./results/llada_summaries_full.csv

Summary Statistics
Total samples processed: 3396
Successful generations: 3396/3396
Average summary length: 601 chars

【Sample Output】:
Reference: We report a case of a ruptured caesarean scar pregnancy in a 29 year-old gravida 5, para 3 with a past obstetric history of two consecutive caesarean sections done 9 and 5 years ago respectively. The ...
Generated: A 29-year-old Cameroonian female with a history of two caesarean sections and a previousopic pregnancy presented at the emergency department with lower abdominal pain and fever. Initial evaluation included consideration for threatened abortion and appendicitis, but after imaging revealed hemoperiton...

✅ All done!


In [3]:
# ============================================
# 检查 CSV 输出质量
# ============================================

import pandas as pd
import os

# 读取 CSV
csv_path = './results/llada_summaries_full.csv'

if not os.path.exists(csv_path):
    # 试试检查点文件
    checkpoint_files = [f for f in os.listdir('./results') if f.startswith('checkpoint_') and f.endswith('.csv')]
    if checkpoint_files:
        latest = sorted(checkpoint_files)[-1]
        csv_path = f'./results/{latest}'
        print(f"使用检查点文件: {latest}")
    else:
        print("❌ 未找到 CSV 文件")
        exit()

df = pd.read_csv(csv_path)
total = len(df)  # 🔥 添加这一行
print(f"✅ 读取成功: {total} 条记录")
print(f"列名: {df.columns.tolist()}\n")

# ============================================
# 1. 检查空值
# ============================================
print("="*60)
print("1. 空值检查")
print("="*60)
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("✅ 无空值")
else:
    for col, count in null_counts.items():
        if count > 0:
            print(f"⚠️ {col}: {count} 个空值")

# ============================================
# 2. 检查生成摘要长度
# ============================================
print("\n" + "="*60)
print("2. 生成摘要长度统计")
print("="*60)

df['gen_len'] = df['predicted_summary'].str.len()
print(f"最短: {df['gen_len'].min()} 字符")
print(f"最长: {df['gen_len'].max()} 字符")
print(f"平均: {df['gen_len'].mean():.0f} 字符")
print(f"中位数: {df['gen_len'].median():.0f} 字符")

# 检查空摘要
empty_count = (df['gen_len'] == 0).sum()
if empty_count > 0:
    print(f"⚠️ 空摘要: {empty_count} 条")
else:
    print("✅ 无空摘要")

# ============================================
# 3. 检查参考摘要长度
# ============================================
print("\n" + "="*60)
print("3. 参考摘要长度统计")
print("="*60)

df['ref_len'] = df['reference_summary'].str.len()
print(f"最短: {df['ref_len'].min()} 字符")
print(f"最长: {df['ref_len'].max()} 字符")
print(f"平均: {df['ref_len'].mean():.0f} 字符")
print(f"中位数: {df['ref_len'].median():.0f} 字符")

# ============================================
# 4. 长度对比
# ============================================
print("\n" + "="*60)
print("4. 生成 vs 参考摘要长度对比")
print("="*60)

df['len_ratio'] = df['gen_len'] / df['ref_len']
print(f"长度比 (生成/参考) 平均值: {df['len_ratio'].mean():.2f}")
print(f"长度比 中位数: {df['len_ratio'].median():.2f}")

# ============================================
# 5. 显示示例
# ============================================
print("\n" + "="*60)
print("5. 示例展示 (前3条)")
print("="*60)

for i in range(min(3, len(df))):
    print(f"\n--- Sample {i+1} ---")
    print(f"Reference ({len(df.iloc[i]['reference_summary'])} chars):")
    print(f"  {df.iloc[i]['reference_summary'][:200]}...")
    print(f"\nGenerated ({len(df.iloc[i]['predicted_summary'])} chars):")
    print(f"  {df.iloc[i]['predicted_summary'][:200]}...")

# ============================================
# 6. 异常值检测
# ============================================
print("\n" + "="*60)
print("6. 异常值检测")
print("="*60)

# 生成摘要过短 (< 20 字符)
short = df[df['gen_len'] < 20]
print(f"过短摘要 (<20字符): {len(short)} 条")

# 生成摘要过长 (> 500 字符)
long = df[df['gen_len'] > 500]
print(f"过长摘要 (>500字符): {len(long)} 条")

# 生成摘要比参考摘要长很多 (> 3倍)
too_long_ratio = df[df['len_ratio'] > 3]
print(f"摘要过长 (生成/参考 > 3): {len(too_long_ratio)} 条")

# 生成摘要比参考摘要短很多 (< 0.3倍)
too_short_ratio = df[df['len_ratio'] < 0.3]
print(f"摘要过短 (生成/参考 < 0.3): {len(too_short_ratio)} 条")

# ============================================
# 7. 最终结论
# ============================================
print("\n" + "="*60)
print("7. 质量评估结论")
print("="*60)

if empty_count == 0 and len(short) < total * 0.05:
    print("✅ 生成摘要整体质量良好")
else:
    print("⚠️ 存在较多空摘要或过短摘要，需要检查")

print(f"\n📁 文件路径: {csv_path}")
print(f"📊 总记录数: {total}")

✅ 读取成功: 3396 条记录
列名: ['id', 'predicted_summary', 'reference_summary']

1. 空值检查
✅ 无空值

2. 生成摘要长度统计
最短: 267 字符
最长: 1009 字符
平均: 601 字符
中位数: 604 字符
✅ 无空摘要

3. 参考摘要长度统计
最短: 121 字符
最长: 3944 字符
平均: 682 字符
中位数: 633 字符

4. 生成 vs 参考摘要长度对比
长度比 (生成/参考) 平均值: 1.13
长度比 中位数: 0.94

5. 示例展示 (前3条)

--- Sample 1 ---
Reference (1167 chars):
  We report a case of a ruptured caesarean scar pregnancy in a 29 year-old gravida 5, para 3 with a past obstetric history of two consecutive caesarean sections done 9 and 5 years ago respectively. The ...

Generated (670 chars):
  A 29-year-old Cameroonian female with a history of two caesarean sections and a previousopic pregnancy presented at the emergency department with lower abdominal pain and fever. Initial evaluation inc...

--- Sample 2 ---
Reference (597 chars):
  A 61-year-old woman of European descent was referred for a new, asymptomatic retinal hemorrhage found on routine examination. Ophthalmoscopy revealed cuticular drusen in both eyes best appreciated on

In [2]:
from huggingface_hub import notebook_login

# 这个会生成一个链接，点击后在浏览器中登录
notebook_login()

In [ ]:
# ============================================
# Llama-3.1-8B 全量推理（3396 条）
# ============================================

import os
import zipfile
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc

# 设置缓存到 /workspace（有 600TB 空间）
os.environ['HF_HOME'] = '/workspace/huggingface_cache'
os.environ['TRANSFORMERS_CACHE'] = '/workspace/huggingface_cache'
os.makedirs('/workspace/huggingface_cache', exist_ok=True)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 输出目录
OUTPUT_DIR = '/workspace/results_llama'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

# ============================================
# Step 1: 加载测试数据
# ============================================
print("\n" + "="*60)
print("Step 1: Loading test data")
print("="*60)

DATA_PATH = 'multiclinsum_test_en.zip'

records = []
with zipfile.ZipFile(DATA_PATH, 'r') as z:
    files = [f for f in z.namelist() if '/fulltext/' in f and f.endswith('.txt')]
    print(f"Found {len(files)} files")

    for file in tqdm(files, desc="Loading"):
        full_text = z.read(file).decode('utf-8')
        summary_file = file.replace('/fulltext/', '/summaries/').replace('.txt', '_sum.txt')
        summary_text = z.read(summary_file).decode('utf-8')
        records.append({'Full_Text': full_text, 'Summary': summary_text})

df_test = pd.DataFrame(records)
print(f"✅ Loaded {len(df_test)} test samples")

# ============================================
# Step 2: 加载模型
# ============================================
print("\n" + "="*60)
print("Step 2: Loading Llama-3.1-8B model")
print("="*60)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.cuda.empty_cache()

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device).eval()

print("✅ Model loaded successfully")

# ============================================
# 提示词
# ============================================
PROMPT_TEMPLATE = """Please generate a concise clinical summary based on the following medical dialogue:

{full_text}

Clinical Summary:"""

def summarize_clinical_text(model, tokenizer, full_text, max_new_tokens=128):
    prompt = PROMPT_TEMPLATE.format(full_text=full_text)
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=2048)
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens, temperature=0.0, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
        )
    generated_ids = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ============================================
# Step 3: 全量推理
# ============================================
print("\n" + "="*60)
print("Step 3: Full inference on all samples")
print("="*60)

total_samples = len(df_test)
print(f"Total samples to process: {total_samples}")

results = []
start_idx = 0

# 检查是否有检查点
import glob
checkpoint_files = glob.glob(f'{OUTPUT_DIR}/checkpoint_*.csv')
if checkpoint_files:
    latest_checkpoint = sorted(checkpoint_files)[-1]
    checkpoint_df = pd.read_csv(latest_checkpoint)
    results = checkpoint_df.to_dict('records')
    start_idx = len(results)
    print(f"✅ Resuming from checkpoint: {start_idx}/{total_samples} already done")

for idx in tqdm(range(start_idx, total_samples), desc="Generating"):
    row = df_test.iloc[idx]
    
    try:
        pred = summarize_clinical_text(model, tokenizer, row['Full_Text'], max_new_tokens=128)
    except Exception as e:
        print(f"\n❌ Error on sample {idx}: {e}")
        pred = ""
    
    results.append({
        'id': idx,
        'full_text': row['Full_Text'],
        'reference_summary': row['Summary'],
        'generated_summary': pred
    })
    
    # 每 100 条保存检查点
    if (idx + 1) % 100 == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv(f'{OUTPUT_DIR}/checkpoint_{idx+1}.csv', index=False)
        print(f"\n📁 Checkpoint saved: {idx+1}/{total_samples}")
        
        # 清理显存
        torch.cuda.empty_cache()
        gc.collect()

# ============================================
# Step 4: 保存最终结果
# ============================================
print("\n" + "="*60)
print("Step 4: Saving final results")
print("="*60)

results_df = pd.DataFrame(results)
csv_path = f'{OUTPUT_DIR}/llama_summaries_full.csv'
results_df.to_csv(csv_path, index=False)
print(f"✅ Full results saved to {csv_path}")

# 统计信息
print("\n" + "="*60)
print("Summary Statistics")
print("="*60)
print(f"Total samples processed: {len(results_df)}")
successful = sum(1 for r in results if r['generated_summary'])
print(f"Successful: {successful}/{len(results_df)}")
avg_len = results_df['generated_summary'].str.len().mean()
print(f"Average summary length: {avg_len:.0f} chars")

if len(results) > 0 and results[0]['generated_summary']:
    print(f"\n【Sample Output】:")
    print(f"Reference: {results[0]['reference_summary'][:200]}...")
    print(f"Generated: {results[0]['generated_summary'][:300]}...")

print("\n✅ All done!")

✅ Output directory: /workspace/results_llama

Step 1: Loading test data
Found 3396 files


Loading: 100%|██████████| 3396/3396 [00:00<00:00, 21626.36it/s]


✅ Loaded 3396 test samples

Step 2: Loading Llama-3.1-8B model
Using device: cuda
GPU: NVIDIA GeForce RTX 4090
GPU Memory: 25.3 GB
Loading meta-llama/Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Generating:  62%|██████▏   | 2100/3396 [1:31:30<58:57,  2.73s/it]  


📁 Checkpoint saved: 2100/3396


Generating:  65%|██████▍   | 2200/3396 [1:35:56<54:22,  2.73s/it]


📁 Checkpoint saved: 2200/3396


Generating:  65%|██████▌   | 2223/3396 [1:36:57<52:02,  2.66s/it]